In [121]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## Load Database

In [122]:
from pathlib import Path
DB_PATH = Path.cwd().parent / "database" / "car_sales.parquet"

# Read database
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
df = pd.read_parquet(DB_PATH)
df.head()

,car_id,date,day,month,year,customer_name,gender,dealer_name,company,model,engine,transmission,color,dealer_no_,body_style,phone,dealer_region,price,income_customer,quantity,discount,gross_sales,discount_amount,sales,cost,total_cost,profit,profit_margin
0,C_CND_000001,2022-01-02,2,1,2022,Geraldine,Male,Buddy Storbeck's Diesel Service Inc,Ford,Expedition,DoubleÂ Overhead Camshaft,Auto,Black,06457-3834,SUV,8264678,Middletown,"467,740,000.00","242,865,000.00",1,0.05,"467,740,000.00","23,387,000.00","444,353,000.00","347,588,502.33","347,588,502.33","96,764,497.67",21.78
1,C_CND_000002,2022-01-02,2,1,2022,Gia,Male,C & M Motors Inc,Dodge,Durango,DoubleÂ Overhead Camshaft,Auto,Black,60504-7114,SUV,6848189,Aurora,"341,810,000.00","26,625,200,000.00",5,0.10,"1,709,050,000.00","170,905,000.00","1,538,145,000.00","281,435,978.53","1,407,179,892.65","130,965,107.35",8.51
2,C_CND_000003,2022-01-02,2,1,2022,Gianna,Male,Capitol KIA,Cadillac,Eldorado,Overhead Camshaft,Manual,Red,38701-8047,Passenger,7298798,Greenville,"566,685,000.00","18,619,650,000.00",2,0.02,"1,133,370,000.00","22,667,400.00","1,110,702,600.00","433,036,322.96","866,072,645.92","244,629,954.08",22.02
3,C_CND_000004,2022-01-02,2,1,2022,Giselle,Male,Chrysler of Tri-Cities,Toyota,Celica,Overhead Camshaft,Manual,Pale White,99301-3882,SUV,6257557,Pasco,"251,860,000.00","242,865,000.00",1,0.50,"251,860,000.00","125,930,000.00","125,930,000.00","190,288,699.80","190,288,699.80","-64,358,699.80",-51.11
4,C_CND_000005,2022-01-02,2,1,2022,Grace,Male,Chrysler Plymouth,Acura,TL,DoubleÂ Overhead Camshaft,Auto,Red,53546-9427,Hatchback,7081483,Janesville,"440,755,000.00","26,355,350,000.00",4,0.10,"1,763,020,000.00","176,302,000.00","1,586,718,000.00","371,011,737.64","1,484,046,950.56","102,671,049.44",6.47


In [123]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23905 entries, 0 to 23904
Data columns (total 28 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   car_id           23905 non-null  object 
 1   date             23905 non-null  object 
 2   day              23905 non-null  int32  
 3   month            23905 non-null  int32  
 4   year             23905 non-null  int32  
 5   customer_name    23905 non-null  object 
 6   gender           23905 non-null  object 
 7   dealer_name      23905 non-null  object 
 8   company          23905 non-null  object 
 9   model            23905 non-null  object 
 10  engine           23905 non-null  object 
 11  transmission     23905 non-null  object 
 12  color            23905 non-null  object 
 13  dealer_no_       23905 non-null  object 
 14  body_style       23905 non-null  object 
 15  phone            23905 non-null  int64  
 16  dealer_region    23905 non-null  object 
 17  price       

## Feature engineering

In [124]:
# Create column feature day_of_week, is_weekend, is_workday
df['date'] = pd.to_datetime(df['date'])
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
df['is_workday'] = df['day_of_week'].apply(lambda x: 1 if x < 5 else 0)

In [125]:
# Build Season feature to define the season of the year based on the month
def season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"
    
df["season"] = df["month"].apply(season)

In [126]:
# Adjust date into dividen specific periods
df["quarter"] = df["date"].dt.quarter
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)

In [127]:
# Build feature price_band as arrangement of price into 4 bands
df["price_band"] = pd.qcut(
    df["price"], q=5, labels=[
        "Budget","Economy", "Mid", "Premium", "Luxury"
    ]
)

In [128]:
# Create a feature of discount_level
df['discount_level'] = pd.cut(
    df['discount'], bins=[0,0.05,0.1,0.15,1],
    labels=[
        "Low",
        "Medium",
        "High",
        "Extreme"
    ]
)

# Create feature of weekend only discount by multiplying is_weekend and discount
df['weekend_discount'] = (df['is_weekend'] * df['discount'])

In [129]:
# Aggregate for revenue featured
daily_sales = (
    df.groupby('date')
        .agg(
            revenue=('sales', 'sum'),
            quantity=('quantity', 'sum'),
            avg_discount=('discount', 'mean'),
            avg_price=('price', 'mean'),
            customers=('customer_name', 'nunique')
        )
        .reset_index()
)

# Transform to revenue
revenue = daily_sales['revenue']

In [130]:
# Create lag features of daily_sales

# Create lg
daily_sales["lag_1"] = daily_sales["revenue"].shift(1)

# Create last week lag
daily_sales["lag_7"] = daily_sales["revenue"].shift(7)

# Create last month lag
daily_sales["lag_30"] = daily_sales["revenue"].shift(30)

In [131]:
# Create features of rolling mean and rolling std for 7 days
daily_sales["rolling_mean_7"] = daily_sales["revenue"].rolling(window=7).mean()
daily_sales["rolling_std_7"] = daily_sales["revenue"].rolling(window=7).std()
daily_sales["rolling_max_7"] = daily_sales["revenue"].rolling(window=7).max()

In [132]:
# Retrieve year, month, day_of_week, quarter, week_of_year, season from df into daily_sales
daily_sales = daily_sales.merge(
    df[['date', 'year', 'month', 'day_of_week', 'quarter', 'week_of_year', 'season']].drop_duplicates(),
    on='date',
    how='left'
)

## Data clean

In [133]:
daily_sales.drop(columns=['date'], inplace=True)
daily_sales.dropna(inplace=True)
daily_sales.sample(10)

,revenue,quantity,avg_discount,avg_price,customers,lag_1,lag_7,lag_30,rolling_mean_7,rolling_std_7,rolling_max_7,year,month,day_of_week,quarter,week_of_year,season
168,"15,768,448,721.20",26,0.07,"617,059,398.67",15,"27,827,733,994.20","34,486,307,930.20","18,697,173,227.60","39,540,485,093.20","30,081,901,843.76","99,470,485,381.40",2022,8,3,3,32,Summer
91,"25,464,934,950.00",73,0.12,"421,480,000.00",35,"14,405,456,520.00","33,142,264,416.10","18,426,328,740.40","26,677,305,079.50","19,114,600,757.77","59,392,303,654.60",2022,5,5,2,18,Spring
211,"44,306,220,310.80",78,0.09,"583,444,998.00",31,"17,507,718,683.00","96,976,630,806.30","30,609,445,300.00","44,335,489,860.90","23,433,842,155.03","90,997,327,787.80",2022,10,2,4,40,Autumn
269,"4,773,394,280.20",14,0.16,"489,691,398.00",5,"29,429,303,998.50","15,338,921,280.20","144,967,887,996.40","60,828,969,660.30","45,052,708,771.97","139,076,533,770.30",2022,12,4,4,49,Winter
394,"28,553,187,580.40",69,0.09,"469,161,210.00",34,"63,348,364,561.30","4,379,763,545.50","98,248,301,270.00","41,995,869,004.20","19,309,072,780.84","64,207,311,323.40",2023,5,5,2,18,Spring
197,"74,138,454,075.00",158,0.12,"539,843,406.00",66,"123,176,004,448.00","65,645,177,724.70","27,827,733,994.20","63,974,525,452.20","37,515,161,345.31","123,176,004,448.00",2022,9,6,3,37,Autumn
223,"25,039,417,480.00",48,0.10,"531,676,460.00",25,"41,110,206,501.00","56,697,011,631.40","39,927,995,450.00","21,123,562,372.90","15,818,368,122.47","42,200,640,127.80",2022,10,3,4,42,Autumn
51,"47,853,487,791.20",107,0.08,"527,447,610.67",45,"73,449,504,537.50","19,378,395,880.20","8,148,570,500.00","32,101,820,167.70","24,992,910,405.78","73,449,504,537.50",2022,3,6,1,11,Spring
165,"56,263,245,746.40",110,0.10,"553,711,331.60",50,"39,335,082,829.00","22,953,831,922.70","20,624,455,600.00","29,678,356,710.40","14,916,963,699.64","56,263,245,746.40",2022,8,0,3,32,Summer
475,"7,873,141,441.20",13,0.03,"583,242,996.00",5,"42,913,830,470.70","20,287,176,921.20","18,261,797,957.20","21,227,210,344.40","12,122,047,711.60","42,913,830,470.70",2023,8,5,3,31,Summer


## Data encoding

In [134]:
from sklearn.preprocessing import LabelEncoder

df_ml = daily_sales.copy()
le = LabelEncoder()

# Encode the object data in dataframe
for col in df_ml.select_dtypes(include=['object', 'category']).columns:
    df_ml[col] = le.fit_transform(df_ml[col])

In [135]:
df_ml.head()

,revenue,quantity,avg_discount,avg_price,customers,lag_1,lag_7,lag_30,rolling_mean_7,rolling_std_7,rolling_max_7,year,month,day_of_week,quarter,week_of_year,season
30,"8,084,202,280.00",26,0.19,"352,605,799.00",10,"11,972,075,150.00","35,694,521,187.50","30,860,360,825.00","18,080,815,858.70","14,198,589,999.91","40,664,374,902.90",2022,2,0,1,8,3
31,"5,499,543,000.00",12,0.12,"581,077,000.00",5,"8,084,202,280.00","5,598,360,990.60","21,199,481,843.40","18,066,699,002.90","14,213,110,732.26","40,664,374,902.90",2022,2,1,1,8,3
32,"11,382,361,151.00",22,0.12,"649,620,699.00",10,"5,499,543,000.00","40,664,374,902.90","12,338,225,620.00","13,883,554,181.20","10,194,848,473.35","36,283,226,127.40",2022,2,2,1,8,3
33,"6,308,733,200.00",20,0.15,"430,860,500.00",10,"11,382,361,151.00","36,283,226,127.40","35,329,496,351.80","9,601,483,763.00","2,912,311,009.65","12,612,860,960.00",2022,2,4,1,8,3
34,"24,148,336,800.00",62,0.11,"447,951,000.00",25,"6,308,733,200.00","11,350,610,600.00","11,695,944,481.20","11,429,730,363.00","6,272,206,173.23","24,148,336,800.00",2022,2,6,1,8,3


## Split data into X and y

In [136]:
from sklearn.model_selection import train_test_split

X = df_ml[["year", "month", "day_of_week", "quarter", "week_of_year", "season", "lag_1", "lag_7", "lag_30",
    "rolling_mean_7", "rolling_std_7", "rolling_max_7", "avg_discount", "avg_price", "customers", "quantity"]]
y = df_ml["revenue"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (465, 16), y_train shape: (465,)


In [137]:
X_train.head(2)

,year,month,day_of_week,quarter,week_of_year,season,lag_1,lag_7,lag_30,rolling_mean_7,rolling_std_7,rolling_max_7,avg_discount,avg_price,customers,quantity
463,2023,7,5,3,29,2,"40,721,192,000.30","23,888,210,754.80","73,974,842,580.80","51,529,022,969.30","26,642,462,243.27","92,001,921,230.10",0.14,"422,102,068.50",59,134
239,2022,11,1,4,45,0,"55,934,360,482.00","85,786,489,207.30","31,056,188,811.20","54,927,754,961.00","42,975,337,360.79","144,967,887,996.40",0.10,"528,816,583.04",107,313


### Machine learning models - Demand prediction (Quantity as a target)

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

models = {
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "CatBoost": CatBoostRegressor(random_state=42, verbose=0),
    "XGBoost": XGBRegressor(random_state=42, verbosity=0, objective='reg:squarederror', tree_method='hist')
}

parameters = {
    "Decision Tree": {
        "max_depth": [None, 5, 10, 15],
        "min_samples_split": [2, 5, 10, 20],
        "min_samples_leaf": [1, 2, 4, 8]
    },
    "Random Forest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [10, 15, 20, None],
        "min_samples_split": [2, 5, 10, 20],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"]
    },
    "CatBoost": {
        "iterations": [200, 300],
        "depth": [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1],
        "l2_leaf_reg": [1, 3, 5, 7]
    },
    "XGBoost": {
        "n_estimators": [200, 300],
        "max_depth": [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.8, 0.9, 1.0],
        "colsample_bytree": [0.8, 0.9, 1.0]
    }
}

# Train and evaluate models
trained_models = {}
results = []
feature_importances = {}

for model_name, model in models.items():
    print("=" * 50)
    print(f"Training {model_name}...")
    print("=" * 50)

    # Machine learning models with hyperparameter tuning using RandomizedSearchCV
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=parameters[model_name],
        n_iter=10,
        cv=5,
        scoring='r2',
        n_jobs=-1,
        verbose=1,
        random_state=42
    )
    random_search.fit(X_train, y_train) # Train the model

    # Get the best model from RandomizedSearchCV
    best_model = random_search.best_estimator_
    print(f"Best parameters for {model_name}: {random_search.best_params_}")
    
    # Store the trained model
    trained_models[model_name] = best_model

    # Best parameters
    print("\nBest Parameters")
    print(random_search.best_params_)

    # Predict on the test set
    y_pred = best_model.predict(X_test)

    # Evaluate the model
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Append results to the list
    results.append({
        "Model": model_name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2
    })

    print(f"{model_name} Evaluation Metrics:")
    print(f"Mean Squared Error (MSE): {mse:.3f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")
    print(f"Mean Absolute Error (MAE): {mae:.3f}")
    print(f"Mean Absolute Percentage Error (MAPE): {mape:.3f}")
    print(f"R-squared (R2): {r2:.3f}")

    # Feature importance
    if hasattr(best_model, 'feature_importances_'):
        importance = pd.DataFrame({
            "Feature": X_train.columns,
            "Importance": best_model.feature_importances_
        }).sort_values(by="Importance", ascending=False)
        feature_importances[model_name] = importance

# ==========================================================
# Comparison Table
# ==========================================================
df_comparison = pd.DataFrame(results).sort_values(by="R2", ascending=False).reset_index(drop=True)
print("\nModel Comparison:")
print(df_comparison)

# Feature importance for each model
for model_name, importance in feature_importances.items():
    print("\n")
    print("=" * 50)
    print(f"{model_name} Feature Importance:")
    print("=" * 50)
    print(importance.head(20))

Training Decision Tree...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters for Decision Tree: {'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 15}

Best Parameters
{'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 15}
Decision Tree Evaluation Metrics:
Mean Squared Error (MSE): 23128982357046788096.000
Root Mean Squared Error (RMSE): 4809260063.362
Mean Absolute Error (MAE): 3061558290.381
Mean Absolute Percentage Error (MAPE): 0.109
R-squared (R2): 0.975


AttributeError: module 'pandas' has no attribute 'DataFraame'

In [ ]:
importance = (
    pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": trained_models["Random Forest"].feature_importances_
    })
    .sort_values("Importance", ascending=False)
)

print(importance)

            Feature  Importance
10  income_customer        0.33
3             model        0.13
2           company        0.08
16     week_of_year        0.08
1       dealer_name        0.07
6             color        0.07
11      day_of_week        0.04
9     dealer_region        0.04
8        body_style        0.04
7        dealer_no_        0.04
5      transmission        0.02
14           season        0.02
4            engine        0.02
0            gender        0.01
15          quarter        0.01
12       is_weekend        0.00
13       is_workday        0.00


## Save models